## 1. Prediction objective

## 2. Combining the data sources

The attendance and booking-event datasets are combined using studio, course, and class start time as the class identifier. The final attendance count is derived from the attendance dataset and used as the prediction target.

In [ ]:
import pandas as pd
from src.data import load_data, prepare_attendance, prepare_booking_events
from pathlib import Path

processed_data_dir = Path("../data/processed")
processed_data_dir.mkdir(parents=True, exist_ok=True)

USE_SYNTHETIC = True

booking_events_raw, attendance_raw = load_data(use_synthetic=USE_SYNTHETIC)

booking_events = prepare_booking_events(booking_events_raw)

attendance = prepare_attendance(attendance_raw)

booking_events["event_order"] = range(len(booking_events))

Missing values are confined to legacy records outside the period covered by both datasets. Since the modeling dataset is constructed only from classes present in both datasets, these records are excluded automatically. The potential duplicate booking events identified during data understanding were verified to represent distinct booking states and are therefore retained.

The final attendance count will be the training label.

In [ ]:
attendance["final_attendance_count"] = attendance["attendance_list"].map(len)

In [ ]:
class_columns=["studio","course","class_start"]
modeling_dataset = attendance[["studio","course","class_start","instructor","final_attendance_count"]].merge(booking_events,
				    on=class_columns,
				    how="inner")

In [ ]:
modeling_dataset.info()

## 3. Creating prediction instances for a chosen prediction horizon

Prediction horizons are defined as the number of hours before class start at which a prediction is made. For each horizon, only booking information available at or before the corresponding prediction time may be used, preventing information leakage from future events.

In [ ]:
PREDICTION_HORIZONS = [24]

In [ ]:
instances_by_horizon = []

for h in PREDICTION_HORIZONS:
   eligible_instances = modeling_dataset.loc[modeling_dataset["event_timestamp"] <= modeling_dataset["class_start"] - pd.Timedelta(hours=h)]

   instances_for_h = ( 
      eligible_instances
      .sort_values(["event_timestamp","event_order"])
      .groupby(class_columns)
      .tail(1)
      .copy()
   )

   instances_for_h["prediction_horizon"] = h

   instances_for_h["prediction_time"] = instances_for_h["class_start"] - pd.Timedelta(hours=h)

   instances_by_horizon.append(instances_for_h)

prediction_instances = pd.concat(
    instances_by_horizon,
    ignore_index=True,
)

Classes without an available snapshot: Booking events are only recorded when a signup or cancellation occurs. The first recorded snapshot may already contain members who booked earlier or hold recurring spots. Therefore, the absence of a snapshot before the prediction time cannot be interpreted as an empty booking state. Class–horizon combinations without a snapshot at or before the prediction time are consequently excluded from the modeling dataset.

## 4. Validating the prediction instances

The constructed instances should satisfy three conditions:

- each class–prediction-horizon combination appears exactly once,
- the selected booking snapshot is available at or before the prediction time,
- the prediction time occurs before class start.

In [ ]:
prediction_instances.duplicated(["studio","course","class_start","prediction_horizon"]).sum()

In [ ]:
(prediction_instances["event_timestamp"] > prediction_instances["prediction_time"]).sum()

In [ ]:
(prediction_instances["prediction_time"] > prediction_instances["class_start"]).sum()

In [ ]:
prediction_instances["snapshot_age_hours"] = (
    prediction_instances["prediction_time"]
    - prediction_instances["event_timestamp"]
).dt.total_seconds() / 3600

prediction_instances["snapshot_age_hours"].describe()

Snapshot recency varies substantially across prediction instances. While the median selected snapshot is approximately 10.5 hours old at prediction time, a small number of instances rely on booking states several days old. This reflects periods without recorded booking activity rather than missing observations.

## 5. Resulting Modeling Dataset

Each row now represents one prediction instance: a class observed at a specific prediction horizon, using the latest booking state available at that time together with the final attendance count as the prediction target.

In [ ]:
prediction_instances.shape


In [ ]:
prediction_instances.info()

In [ ]:
prediction_instances.to_parquet(
    PROCESSED_DATA_DIR / "prediction_instances.parquet",
    index=False,
)